In [1]:
import pandas as pd

# Load the existing Night Parrot baseline
night_parrot_df = pd.read_parquet("pilot_bird_occurrences_clean.parquet")

# Define the new raw CSV files
raw_csv_files = [
    "plains_wanderer_raw.csv",
    "princess_parrot_raw.csv",
    "dusky_grasswren_raw.csv",
    "malleefowl_raw.csv",
    "rufous_scrub_bird_raw.csv"
]

# Start the master list with the Night Parrot baseline
cleaned_dataframes = [night_parrot_df] 

# Batch process each new CSV
for file in raw_csv_files:
    # Load the raw data
    df = pd.read_csv(file)
    
    # Drop rows missing critical GPS coordinates
    df = df.dropna(subset=['decimalLatitude', 'decimalLongitude'])
    
    # Apply Australian continental bounding box filter
    df = df[
        (df['decimalLatitude'] >= -44.0) & (df['decimalLatitude'] <= -10.0) &
        (df['decimalLongitude'] >= 112.0) & (df['decimalLongitude'] <= 154.0)
    ]
    
    # Filter by spatial precision (keep uncertainty < 10000m or where not specified)
    # Filling NaN with 0 ensures we don't accidentally drop valid records missing an uncertainty value
    df = df[df['coordinateUncertaintyInMeters'].fillna(0) < 10000]
    
    # Append the cleaned DataFrame to the master list
    cleaned_dataframes.append(df)

# 4. Master Merge
master_df = pd.concat(cleaned_dataframes, ignore_index=True)

# Verify the output 
print(f"Total aggregated valid records: {len(master_df)}")
print(master_df['scientificName'].value_counts())

/var/folders/6q/8vs17lln4qj3g7qfr71dx5l00000gn/T/ipykernel_49032/2765108208.py:21: DtypeWarning: Columns (2,7,9,10,11,12,27,29,31,32,34,35,36,38,39,46,48,60,63,66,67,68,69,70,75,77,80,93,94,95,100,104,125,127,129,130,131,140,143,146,161,163,170,175,179,182,189,190) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/var/folders/6q/8vs17lln4qj3g7qfr71dx5l00000gn/T/ipykernel_49032/2765108208.py:21: DtypeWarning: Columns (2,3,4,6,7,8,9,10,11,12,13,14,16,22,27,28,29,30,31,32,34,35,36,37,38,39,41,46,47,48,49,58,60,62,63,65,66,67,68,69,70,74,75,77,80,87,93,94,95,97,100,101,102,103,104,125,126,127,129,130,131,140,143,146,153,154,155,157,160,161,163,165,166,170,172,175,179,181,182,189,190) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/var/folders/6q/8vs17lln4qj3g7qfr71dx5l00000gn/T/ipykernel_49032/2765108208.py:21: DtypeWarning: Columns (2,7,9,10,11,12,29,31,32,34,35,36,38,39,44,46,48,49,57,60

Total aggregated valid records: 20626
scientificName
Leipoa ocellata                                  9445
Pedionomus torquatus                             5273
Amytornis (Amytornis) purnelli                   3213
Atrichornis (Atrichornis) rufescens              2539
Pezoporus occidentalis                             55
Polytelis alexandrae                               55
Atrichornis (Atrichornis) rufescens rufescens      38
Atrichornis (Atrichornis) rufescens ferrieri        8
Name: count, dtype: int64


In [3]:
# Standardize the scientific names for the frontend UI
name_mapping = {
    'Amytornis (Amytornis) purnelli': 'Amytornis purnelli',
    'Atrichornis (Atrichornis) rufescens': 'Atrichornis rufescens',
    'Atrichornis (Atrichornis) rufescens rufescens': 'Atrichornis rufescens',
    'Atrichornis (Atrichornis) rufescens ferrieri': 'Atrichornis rufescens'
}

master_df['scientificName'] = master_df['scientificName'].replace(name_mapping)

# Verify the clean rollup
print(f"Total aggregated valid records: {len(master_df)}")
print(master_df['scientificName'].value_counts())

Total aggregated valid records: 20626
scientificName
Leipoa ocellata           9445
Pedionomus torquatus      5273
Amytornis purnelli        3213
Atrichornis rufescens     2585
Pezoporus occidentalis      55
Polytelis alexandrae        55
Name: count, dtype: int64


In [4]:
import geopandas as gpd

# Convert to spatial geometries using Longitude (X) and Latitude (Y)
geometry = gpd.points_from_xy(master_df['decimalLongitude'], master_df['decimalLatitude'])

# Initialize the GeoDataFrame with standard WGS 84 GPS coordinates
gdf = gpd.GeoDataFrame(master_df, geometry=geometry, crs="EPSG:4326")

# Reproject to Australian Albers (EPSG:3577) 
gdf = gdf.to_crs("EPSG:3577")

# Strip the bloat (ALA includes 200+ columns, we only need the essentials)
columns_to_keep = ['scientificName', 'eventDate', 'year', 'geometry']
existing_cols = [col for col in columns_to_keep if col in gdf.columns]
final_gdf = gdf[existing_cols]

# Export the unified jumbo dataset
final_gdf.to_parquet("multispecies_occurrences_clean.parquet")

print("Phase 2 Complete. Master spatial file generated.")

Phase 2 Complete. Master spatial file generated.
